In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"\pypipeline\data\raw\static\hourly\aqi\limited\us_paro_hourly.csv")

In [3]:
df[df["value"]<0]["value"].count()

np.int64(3980)

### Feature Extraction


In [4]:
df_pm25 = df[df["parameter"]=="pm25"].\
drop(columns=["locationId","location","parameter","unit","utc","country","latitude","longitude","city"]).\
rename(columns={"value":"pm25","local":"date"}).reset_index(drop=True)

df_o3 = df[df["parameter"]=="o3"].\
drop(columns=["locationId","location","parameter","unit","utc","country","latitude","longitude","city"]).\
rename(columns={"value":"o3","local":"date"}).reset_index(drop=True)

In [5]:
df_merged = df_pm25.merge(df_o3, on="date", how="outer").sort_values("date").reset_index(drop=True)

### Remove negative values

In [6]:
df_merged.loc[df_merged["pm25"]<0, "pm25"]=pd.NA
df_merged.loc[df_merged["o3"]<0, "o3"]=pd.NA

In [7]:
df_merged["date"] = pd.to_datetime(df_merged["date"])
df_merged.set_index("date", inplace=True)

In [8]:
df_merged

,pm25,o3
date,,
2017-03-03 05:00:00+05:45,106.1,0.002
2017-03-03 06:00:00+05:45,134.5,0.002
2017-03-03 07:00:00+05:45,154.7,0.002
2017-03-03 08:00:00+05:45,155.4,0.003
2017-03-03 09:00:00+05:45,178.7,0.005
...,...,...
2021-03-12 20:00:00+05:45,58.0,0.039
2021-03-12 21:00:00+05:45,58.0,0.030
2021-03-12 22:00:00+05:45,60.0,0.030


In [9]:
df_merged.sort_index

<bound method DataFrame.sort_index of                             pm25     o3
date                                   
2017-03-03 05:00:00+05:45  106.1  0.002
2017-03-03 06:00:00+05:45  134.5  0.002
2017-03-03 07:00:00+05:45  154.7  0.002
2017-03-03 08:00:00+05:45  155.4  0.003
2017-03-03 09:00:00+05:45  178.7  0.005
...                          ...    ...
2021-03-12 20:00:00+05:45   58.0  0.039
2021-03-12 21:00:00+05:45   58.0  0.030
2021-03-12 22:00:00+05:45   60.0  0.030
2021-03-12 23:00:00+05:45   69.0  0.030
2021-03-13 00:00:00+05:45   69.0  0.051

[32364 rows x 2 columns]>

### time delta check
from the result we can see most of the data is hourly but some have larger deltas, we need to force everything to hourly

In [10]:
deltas = df_merged.index.sort_values().diff().value_counts()
deltas.head()

date
0 days 01:00:00    31634
0 days 02:00:00      515
0 days 03:00:00      109
0 days 04:00:00       44
0 days 05:00:00       19
Name: count, dtype: int64

In [11]:
df_merged= df_merged.asfreq('H') # force to hourly frequency

C:\Users\ZENBOOK\AppData\Local\Temp\ipykernel_11308\1743174348.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_merged= df_merged.asfreq('H') # force to hourly frequency


In [12]:
df_merged["hour"] = df_merged.index.hour
df_merged["day_of_week"] = df_merged.index.dayofweek
df_merged["is_weekend"] = df_merged["day_of_week"].isin([5,6]).astype(int)
df_merged["is_night"]=df_merged["hour"].isin([0,1,2,3,4,5]).astype(int)

### Finding gaps in time series data
use of flag features that need to be encoded during later model training
#### Rules used
The gap in this case is the gap in the sensor data not the gap on time
- Small Gap: <= 6 Hours
- Medium Gap: > 6 Hours and <= 12
- Large Gap: > 12 Hours
- very large Gap >24 Hours
#### Gap Imputation Methods
##### Small Gap
use linear interpolation, or Kalman filter
##### Medium Gap
use KNN imputation + flag
##### Large Gap
donot impute, just leave it as it is
##### Very large gap
use segmentation methods so that data doesn't leak for lag features




In [13]:
df_merged

,pm25,o3,hour,day_of_week,is_weekend,is_night
date,,,,,,
2017-03-03 05:00:00+05:45,106.1,0.002,5,4,0,1
2017-03-03 06:00:00+05:45,134.5,0.002,6,4,0,0
2017-03-03 07:00:00+05:45,154.7,0.002,7,4,0,0
2017-03-03 08:00:00+05:45,155.4,0.003,8,4,0,0
2017-03-03 09:00:00+05:45,178.7,0.005,9,4,0,0
...,...,...,...,...,...,...
2021-03-12 20:00:00+05:45,58.0,0.039,20,4,0,0
2021-03-12 21:00:00+05:45,58.0,0.030,21,4,0,0
2021-03-12 22:00:00+05:45,60.0,0.030,22,4,0,0


In [14]:
df_merged["pm25_missing"] = df_merged["pm25"].isna().astype(int)
df_merged["o3_missing"] = df_merged["o3"].isna().astype(int) 

In [15]:
pm25_run_id = (df_merged["pm25_missing"] != df_merged["pm25_missing"].shift()).cumsum()
o3_run_id = (df_merged["o3_missing"] != df_merged["o3_missing"].shift()).cumsum()

In [16]:
pm25_gap_length = (df_merged["pm25_missing"].groupby(pm25_run_id).transform("sum"))
o3_gap_length = (df_merged["o3_missing"].groupby(o3_run_id).transform("sum"))

In [17]:
df_merged["pm25_gap_length"] = pm25_gap_length
df_merged["o3_gap_length"] = o3_gap_length

In [18]:
SMALL_GAP_THRESHOLD = 6
MEDIUM_GAP_THRESHOLD = 12
VERY_LARGE_GAP_THRESHOLD = 24   

In [19]:
df_merged.isna().sum()

pm25               5915
o3                 6689
hour                  0
day_of_week           0
is_weekend            0
is_night              0
pm25_missing          0
o3_missing            0
pm25_gap_length       0
o3_gap_length         0
dtype: int64

#### Small Gap case (<=6 hours)
Used time based interpolation from pandas

In [20]:
df_imputation = df_merged.copy()

In [21]:
small_gap_mask_pm25 = df_merged["pm25_gap_length"] <= SMALL_GAP_THRESHOLD
small_gap_mask_o3 = df_merged["o3_gap_length"] <= SMALL_GAP_THRESHOLD

df_imputation.loc[small_gap_mask_pm25, "pm25"] = df_imputation["pm25"].\
interpolate(method="time",limit=SMALL_GAP_THRESHOLD)
df_imputation.loc[small_gap_mask_pm25, "was_imputed"] = int(1)
df_imputation.loc[small_gap_mask_pm25,"imputation_confidence"]="high"

df_imputation.loc[small_gap_mask_o3, "o3"] = df_imputation["o3"].\
interpolate(method="time",limit=SMALL_GAP_THRESHOLD)
df_imputation.loc[small_gap_mask_o3, "was_imputed"] = int(1)
df_imputation.loc[small_gap_mask_o3,"imputation_confidence"]="high"



#### Medium imputation case (>6 and <=12 hours)
using KNN imputation and lag freatures

In [22]:
from sklearn.impute import KNNImputer
import numpy as np

medium_gap_mask_pm25 = (df_merged["pm25_gap_length"] > SMALL_GAP_THRESHOLD) & \
                        (df_merged["pm25_gap_length"] <= MEDIUM_GAP_THRESHOLD)
medium_gap_mask_o3 = (df_merged["o3_gap_length"] > SMALL_GAP_THRESHOLD) & \
                        (df_merged["o3_gap_length"] <= MEDIUM_GAP_THRESHOLD)


#cyclic features
df_imputation["hour_sin"]=np.sin(2 * np.pi * df_imputation['hour']/24)
df_imputation["hour_cos"]=np.cos(2 * np.pi * df_imputation['hour']/24)
cols = ['pm25','o3','hour_sin','hour_cos']

imp = KNNImputer(n_neighbors=6)
imputed_knn= imp.fit_transform(df_imputation[cols])
df_all_imputed = pd.DataFrame(imputed_knn, columns=cols, index=df_imputation.index)
                               
                               



In [23]:
df_all_imputed

,pm25,o3,hour_sin,hour_cos
date,,,,
2017-03-03 05:00:00+05:45,106.1,0.002,0.965926,2.588190e-01
2017-03-03 06:00:00+05:45,134.5,0.002,1.000000,6.123234e-17
2017-03-03 07:00:00+05:45,154.7,0.002,0.965926,-2.588190e-01
2017-03-03 08:00:00+05:45,155.4,0.003,0.866025,-5.000000e-01
2017-03-03 09:00:00+05:45,178.7,0.005,0.707107,-7.071068e-01
...,...,...,...,...
2021-03-12 20:00:00+05:45,58.0,0.039,-0.866025,5.000000e-01
2021-03-12 21:00:00+05:45,58.0,0.030,-0.707107,7.071068e-01
2021-03-12 22:00:00+05:45,60.0,0.030,-0.500000,8.660254e-01


In [24]:
df_imputation.loc[medium_gap_mask_pm25,"pm25"]= df_all_imputed.loc[medium_gap_mask_pm25,"pm25"]
df_imputation.loc[medium_gap_mask_pm25,"was_imputed"]=int(1)
df_imputation.loc[medium_gap_mask_pm25,"imputation_confidence"]="medium"

df_imputation.loc[medium_gap_mask_o3,"o3"]= df_all_imputed.loc[medium_gap_mask_o3,"o3"]
df_imputation.loc[medium_gap_mask_o3,"was_imputed"]=int(1)
df_imputation.loc[medium_gap_mask_o3,"imputation_confidence"]="medium"



In [25]:
df_imputation

,pm25,o3,hour,day_of_week,is_weekend,is_night,pm25_missing,o3_missing,pm25_gap_length,o3_gap_length,was_imputed,imputation_confidence,hour_sin,hour_cos
date,,,,,,,,,,,,,,
2017-03-03 05:00:00+05:45,106.1,0.002,5,4,0,1,0,0,0,0,1.0,high,0.965926,2.588190e-01
2017-03-03 06:00:00+05:45,134.5,0.002,6,4,0,0,0,0,0,0,1.0,high,1.000000,6.123234e-17
2017-03-03 07:00:00+05:45,154.7,0.002,7,4,0,0,0,0,0,0,1.0,high,0.965926,-2.588190e-01
2017-03-03 08:00:00+05:45,155.4,0.003,8,4,0,0,0,0,0,0,1.0,high,0.866025,-5.000000e-01
2017-03-03 09:00:00+05:45,178.7,0.005,9,4,0,0,0,0,0,0,1.0,high,0.707107,-7.071068e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-03-12 20:00:00+05:45,58.0,0.039,20,4,0,0,0,0,0,0,1.0,high,-0.866025,5.000000e-01
2021-03-12 21:00:00+05:45,58.0,0.030,21,4,0,0,0,0,0,0,1.0,high,-0.707107,7.071068e-01
2021-03-12 22:00:00+05:45,60.0,0.030,22,4,0,0,0,0,0,0,1.0,high,-0.500000,8.660254e-01


In [26]:
df_imputation.isna().sum()

pm25                     3867
o3                       4392
hour                        0
day_of_week                 0
is_weekend                  0
is_night                    0
pm25_missing                0
o3_missing                  0
pm25_gap_length             0
o3_gap_length               0
was_imputed              3392
imputation_confidence    3392
hour_sin                    0
hour_cos                    0
dtype: int64

#### Large gaps >12, <=24 hours
leaving nan but flagging

In [27]:
large_gap_mask_pm25 = (df_merged["pm25_gap_length"] > MEDIUM_GAP_THRESHOLD) & (df_merged["pm25_gap_length"] <= VERY_LARGE_GAP_THRESHOLD)
large_gap_mask_o3 = (df_merged["o3_gap_length"] > MEDIUM_GAP_THRESHOLD) & (df_merged["o3_gap_length"] <= VERY_LARGE_GAP_THRESHOLD)

df_imputation.loc[large_gap_mask_pm25,"imputation_confidence"]= "low"
df_imputation.loc[large_gap_mask_o3,"imputation_confidence"]= "low"

#### Very large gaps >24 hours
using segmentation to avoid data leakage in lag features

In [28]:
VERY_LARGE_GAP_THRESHOLD = 24
segment = (
    (df_imputation['pm25_missing'].shift(fill_value=0) == 1) & (df_imputation['pm25_gap_length'].shift(fill_value=0) > VERY_LARGE_GAP_THRESHOLD)
) | (
    (df_imputation['o3_missing'].shift(fill_value=0) == 1) & (df_imputation['o3_gap_length'].shift(fill_value=0) > VERY_LARGE_GAP_THRESHOLD)
)

In [29]:
df_imputation["segment_id"]=segment.cumsum()


#### reset "was_missing" index

In [33]:

df_imputation["pm25_missing"] = df_imputation["pm25"].isna().astype(int)
df_imputation["o3_missing"] = df_imputation["o3"].isna().astype(int)

### Feature Engineering with segmentation

In [ ]:
df_engineering = df_imputation.copy()

lags=[i for i in range(1,25)]

#lag freatures
for lag in lags:
    df_engineering[f"pm25_lag_{lag}"]=df_engineering.groupby("segment_id")["pm25"].shift(lag)# grouping by segment to avoid gap leak
    df_engineering[f"o3_lag_{lag}"]=df_engineering.groupby("segment_id")["o3"].shift(lag)




#### rolling mean

In [ ]:
df_engineering['pm25_roll_3h']  = (df_engineering.groupby('segment_id')['pm25']
                        .apply(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()) # value is shifted by 1 to only caputre past
                        .reset_index(level=0, drop=True))

df_engineering["pm25_roll_6"]= (df_engineering.groupby('segment_id')['pm25']
                        .apply(lambda s: s.shift(1).rolling(window=6, min_periods=1).mean())
                        .reset_index(level=0, drop=True))

df_engineering['pm25_roll_12'] = (df_engineering.groupby('segment_id')['pm25']
                        .apply(lambda s: s.shift(1).rolling(window=12, min_periods=1).mean())
                        .reset_index(level=0, drop=True))

df_engineering['pm25_roll_24h'] = (df_engineering.groupby('segment_id')['pm25']
                        .apply(lambda s: s.shift(1).rolling(window=24, min_periods=1).mean())
                        .reset_index(level=0, drop=True))


df_engineering['o3_roll_3h']  = (df_engineering.groupby('segment_id')['o3']
                        .apply(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())
                        .reset_index(level=0, drop=True))   
df_engineering["o3_roll_6"]= (df_engineering.groupby('segment_id')['o3']
                        .apply(lambda s: s.shift(1).rolling(window=6, min_periods=1).mean())
                        .reset_index(level=0, drop=True))   
df_engineering['o3_roll_12'] = (df_engineering.groupby('segment_id')['o3']       
                        .apply(lambda s: s.shift(1).rolling(window=12, min_periods=1).mean())
                        .reset_index(level=0, drop=True))
df_engineering['o3_roll_24h'] = (df_engineering.groupby('segment_id')['o3']               
                        .apply(lambda s: s.shift(1).rolling(window=24, min_periods=1).mean())
                        .reset_index(level=0, drop=True))

#### Momentum

In [41]:
import numpy as np

def trend_12(x):
    t = np.arange(len(x))
    return np.polyfit(t, x, 1)[0]

df_engineering["pm25_slope_12h"] = df_engineering.groupby("segment_id")["pm25"].\
    rolling(window=12,min_periods=12).apply(trend_12)

df_engineering["o3_slope_12h"] = df_engineering.groupby("segment_id")["o3"].\
    rolling(window=12,min_periods=12).apply(trend_12)

In [ ]:
df_enginnering

#### Interactions

In [42]:
df_engineering.groupby("segment_id")["pm25"].apply(lambda x: x / df_engineering.groupby("segment_id")["o3"])

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (4503, 2) + inhomogeneous part.